# Vibely: Klasifikasi Mood Lagu

Alur notebook ini:
1. Memuat dataset **Moodify** (fitur audio dan label mood).
2. Melatih model Random Forest dan mengevaluasinya.
3. Mengekspor model ke Core ML (`.mlmodel`).
4. Melabeli katalog lagu yang punya judul dan artis.
5. Mengekspor `songs.json` untuk aplikasi SwiftUI.

Struktur folder yang diharapkan:
```
proyek/
  vibely.ipynb
  dataset/
    278k_song_labelled.csv   (Moodify)
    dataset.csv              (katalog lagu: judul, artis, fitur audio)
```


In [1]:
# Jalankan sekali jika library belum terpasang (lalu restart kernel).
# Pakai Python 3.12.
# %pip install pandas scikit-learn==1.5.1 coremltools

## 1. Pengaturan

In [2]:
from pathlib import Path
import json
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

DATASET_DIR = Path("dataset")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Ganti nama file di sini jika berbeda dengan milikmu
MOODIFY_FILE = "278k_song_labelled.csv"
CATALOG_FILE = "dataset.csv"

# Urutan label sesuai dokumentasi Moodify
MOODS = {0: "sad", 1: "happy", 2: "energetic", 3: "calm"}

# spec_rate dan kolom nomor baris sengaja tidak dipakai
FEATURES = [
    "duration_ms", "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]

# Jika nama kolom katalogmu berbeda, ubah di sini (kiri = nama di CSV-mu)
COLUMN_MAP = {
    # "name": "track_name",
    # "id": "track_id",
    # "artist_name": "artists",
    # "duration (ms)": "duration_ms",
}

print("File di folder dataset:")
for f in sorted(DATASET_DIR.glob("*.csv")):
    print(" -", f.name)

File di folder dataset:
 - 278k_labelled_uri.csv
 - 278k_song_labelled.csv


## 2. Memuat dataset Moodify

In [3]:
moodify = pd.read_csv(DATASET_DIR / MOODIFY_FILE)
moodify = moodify.rename(columns={"duration (ms)": "duration_ms"})
moodify = moodify[FEATURES + ["labels"]].dropna()

print("Jumlah baris:", len(moodify))
moodify["labels"].map(MOODS).value_counts()

Jumlah baris: 277938


labels
happy        106429
sad           82058
energetic     47065
calm          42386
Name: count, dtype: int64

In [4]:
# Rata-rata fitur per mood (untuk memastikan arti label masuk akal)
moodify.groupby(moodify["labels"].map(MOODS))[
    ["valence", "energy", "acousticness", "tempo", "danceability"]
].mean().round(2)

,valence,energy,acousticness,tempo,danceability
labels,,,,,
calm,0.22,0.18,0.84,106.27,0.39
energetic,0.44,0.87,0.03,134.00,0.50
happy,0.60,0.69,0.21,121.19,0.68
sad,0.37,0.40,0.58,114.80,0.50


In [17]:

# 1. Periksa duplikasi seluruh fitur audio
print(
    "Duplikasi fitur:",
    moodify.duplicated(subset=FEATURES).sum()
)

# 2. Periksa apakah fitur audio yang sama
# memiliki label berbeda
label_counts = moodify.groupby(FEATURES)["labels"].nunique()

print(
    "Kombinasi fitur dengan label berbeda:",
    (label_counts > 1).sum()
)

# 3. Periksa distribusi label
print(moodify["labels"].value_counts(normalize=True).round(3))


Duplikasi fitur: 1678
Kombinasi fitur dengan label berbeda: 0
labels
1    0.383
0    0.295
2    0.169
3    0.153
Name: proportion, dtype: float64


## 3. Melatih dan mengevaluasi model

In [5]:
X = moodify[FEATURES]
y = moodify["labels"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# max_depth dibatasi agar ukuran model Core ML tetap kecil
model = RandomForestClassifier(
    n_estimators=60, max_depth=14, min_samples_leaf=5,
    n_jobs=-1, random_state=42,
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(f"Akurasi: {accuracy_score(y_test, pred):.3f}\n")
print(classification_report(
    y_test, pred, target_names=[MOODS[i] for i in range(4)]
))

Akurasi: 0.932

              precision    recall  f1-score   support

         sad       0.95      0.93      0.94     16412
       happy       0.91      0.94      0.93     21286
   energetic       0.92      0.87      0.90      9413
        calm       0.97      0.97      0.97      8477

    accuracy                           0.93     55588
   macro avg       0.94      0.93      0.93     55588
weighted avg       0.93      0.93      0.93     55588



In [18]:
# Cek apakah kombinasi fitur test juga muncul di training
train_features = X_train[FEATURES].copy()
test_features = X_test[FEATURES].copy()

train_keys = pd.MultiIndex.from_frame(train_features)
test_keys = pd.MultiIndex.from_frame(test_features)

overlap_mask = test_keys.isin(train_keys)

print("Jumlah data test:", len(test_features))
print("Data test dengan fitur yang juga ada di train:",
      overlap_mask.sum())
print("Persentase overlap:",
      round(overlap_mask.mean() * 100, 2), "%")

Jumlah data test: 55588
Data test dengan fitur yang juga ada di train: 544
Persentase overlap: 0.98 %


In [19]:
# Gabungkan fitur dan label dari data training
train_check = X_train[FEATURES].copy()
train_check["labels"] = y_train

# Ambil fitur dan label dari data test
test_check = X_test[FEATURES].copy()
test_check["labels"] = y_test

# Cek kombinasi fitur yang muncul di kedua set
matched = test_check.merge(
    train_check.drop_duplicates(),
    on=FEATURES,
    how="inner",
    suffixes=("_test", "_train")
)

print("Jumlah pasangan fitur yang cocok:", len(matched))
print(
    "Label sama:",
    (matched["labels_test"] == matched["labels_train"]).sum()
)
print(
    "Label berbeda:",
    (matched["labels_test"] != matched["labels_train"]).sum()
)

Jumlah pasangan fitur yang cocok: 544
Label sama: 544
Label berbeda: 0


In [16]:

# Bandingkan performa training dan test
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print(f"Training Accuracy : {train_acc:.4f}")
print(f"Test Accuracy     : {test_acc:.4f}")
print(f"Gap               : {train_acc - test_acc:.4f}")


Training Accuracy : 0.9620
Test Accuracy     : 0.9322
Gap               : 0.0298


In [6]:
# Confusion matrix (baris = label asli, kolom = prediksi)
names = [MOODS[i] for i in range(4)]
pd.DataFrame(confusion_matrix(y_test, pred), index=names, columns=names)

,sad,happy,energetic,calm
sad,15308,806,113,185
happy,570,20102,590,24
energetic,94,1110,8203,6
calm,187,78,4,8208


In [7]:
# Fitur mana yang paling berpengaruh
pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

energy              0.234
instrumentalness    0.213
acousticness        0.194
danceability        0.117
loudness            0.087
speechiness         0.054
valence             0.051
duration_ms         0.028
tempo               0.015
liveness            0.008
dtype: float64

## 4. Ekspor ke Core ML

Konversi bergantung pada versi library. Jika gagal (misalnya di Windows),
lewati saja. `songs.json` tetap dibuat, dan model bisa dibuat lewat Create ML di Mac.

In [8]:
try:
    import coremltools as ct
    mlmodel = ct.converters.sklearn.convert(model, FEATURES, "mood")
    mlmodel.short_description = "Klasifikasi mood lagu dari fitur audio"
    mlmodel.save(str(OUTPUT_DIR / "MoodClassifier.mlmodel"))
    print("Core ML tersimpan di output/MoodClassifier.mlmodel")
except Exception as e:
    print("Konversi Core ML gagal:", e)

Konversi Core ML gagal: No module named 'coremltools'


## 5. Memuat katalog lagu (judul dan artis)

In [11]:
catalog = pd.read_csv(DATASET_DIR / CATALOG_FILE).rename(columns=COLUMN_MAP)

needed = FEATURES + ["track_id", "track_name", "artists"]
missing = [c for c in needed if c not in catalog.columns]
if missing:
    print("Kolom di file katalogmu:", list(catalog.columns))
    raise SystemExit(f"Kolom ini tidak ditemukan: {missing}. Isi COLUMN_MAP di bagian Pengaturan.")

catalog = (catalog.dropna(subset=needed)
                  .drop_duplicates(subset="track_id")
                  .drop_duplicates(subset=["track_name", "artists"]))
print("Jumlah lagu setelah dibersihkan:", len(catalog))
catalog[["track_name", "artists"]].head()

Jumlah lagu setelah dibersihkan: 81343


,track_name,artists
0,Comedy,Gen Hoshino
1,Ghost - Acoustic,Ben Woodward
2,To Begin Again,Ingrid Michaelson;ZAYN
3,Can't Help Falling In Love,Kina Grannis
4,Hold On,Chord Overstreet


## 6. Memberi label mood dengan model

Hanya prediksi dengan keyakinan di atas `MIN_CONF` yang dipakai, supaya lagu yang moodnya meragukan tidak ikut.

In [12]:
PER_MOOD = 250     # jumlah lagu per mood di aplikasi
MIN_CONF = 0.6     # batas minimal keyakinan model

proba = model.predict_proba(catalog[FEATURES])
catalog = catalog.assign(mood_id=proba.argmax(axis=1), confidence=proba.max(axis=1))
catalog["mood"] = catalog["mood_id"].map(MOODS)

print("Sebaran mood di katalog (semua lagu):")
print(catalog["mood"].value_counts(), "\n")

confident = catalog[catalog["confidence"] >= MIN_CONF]
print("Lagu dengan keyakinan >=", MIN_CONF, ":", len(confident))

Sebaran mood di katalog (semua lagu):
mood
happy        34743
sad          21880
energetic    18569
calm          6151
Name: count, dtype: int64 

Lagu dengan keyakinan >= 0.6 : 73886


In [13]:
parts = []
for mood_id, name in MOODS.items():
    sub = confident[confident["mood_id"] == mood_id]
    sub = sub.nlargest(PER_MOOD * 3, "confidence")
    picked = sub.sample(min(PER_MOOD, len(sub)), random_state=42)
    parts.append(picked)
    print(f"{name}: {len(sub)} kandidat, diambil {len(picked)}")

selected = pd.concat(parts)
len(selected)

sad: 750 kandidat, diambil 250
happy: 750 kandidat, diambil 250
energetic: 750 kandidat, diambil 250
calm: 750 kandidat, diambil 250


1000

## 7. Ekspor `songs.json` untuk SwiftUI

In [14]:
songs = [
    {
        "id": r.track_id,
        "title": r.track_name,
        "artist": str(r.artists).replace(";", ", "),
        "mood": r.mood,
        "confidence": round(float(r.confidence), 3),
        "spotifyURL": f"https://open.spotify.com/track/{r.track_id}",
    }
    for r in selected.itertuples()
]

with open(OUTPUT_DIR / "songs.json", "w", encoding="utf-8") as f:
    json.dump(songs, f, ensure_ascii=False, indent=2)

print(len(songs), "lagu ditulis ke output/songs.json")

1000 lagu ditulis ke output/songs.json


## 8. Cek manual (untuk laporan)

Dengarkan beberapa lagu di bawah dan nilai apakah moodnya masuk akal.
Hasil penilaian ini bisa dipakai sebagai evaluasi manual di bab pengujian.

In [15]:
check = selected.groupby("mood").sample(8, random_state=1)
check[["mood", "track_name", "artists", "confidence"]].sort_values("mood").reset_index(drop=True)

,mood,track_name,artists,confidence
0,calm,Still Life,Alexis Ffrench,1.000000
1,calm,Baby Lullaby,Piano Peace,1.000000
2,calm,Ultraviolet,James Heather,0.999994
3,calm,"Die Zauberflöte, K. 620 / Zweiter Aufzug: Mars...",Wolfgang Amadeus Mozart;Berliner Philharmonike...,0.999994
4,calm,Almost June,Ludovico Einaudi,1.000000
5,calm,Breezin',Jane Leslie,1.000000
6,calm,"Baby Mine (from ""Dumbo"")",The Sleep Diaries,0.999994
7,calm,First Love,Smyang Piano,1.000000
8,energetic,Vultures of North,Orbit Culture,0.999646
9,energetic,Times Like These,Foo Fighters,0.999545
